# <span style="color:blue; font-weight:bold;">1. Importation des données</span>

La première étape consiste à importer le dataset contenant les informations des vols (retard au départ, retard à l’arrivée, jour de la semaine, heure prévue, aéroport d’origine, aéroport de destination, etc.).

Nous utilisons généralement Pandas pour charger les données :

Chargement du fichier CSV

Vérification des premières lignes du dataset

Vérification de la présence de valeurs manquantes

<h4>Objectif : préparer les données brutes pour le prétraitement.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


df = pd.read_csv('data.csv')
print("Dimensions des données:", df.shape)
df.head()


Dimensions des données: (781081, 15)


,depdelay,arrdelay,origin,dest,uniquecarrier,scheduledhour,tailnum,numflights,origincityname,windspeed,windgustdummy,raindummy,snowdummy,is_holiday_season,est_retarde
0,-2,-11.0,5.875096,5.678700,5.262673,17,4.984410,15.471000,5.875096,9.133333,1,0,0,False,0
1,-10,-1.0,7.817186,2.527276,8.747081,8,9.403636,20.132999,5.577255,13.000000,0,0,0,False,0
2,23,13.0,5.994635,3.972538,8.646316,8,10.882033,17.000999,5.994635,7.166667,1,0,0,False,0
3,-4,-14.0,-0.443869,3.599142,6.162031,7,6.645973,17.858999,-0.443869,21.000000,0,0,0,False,0
4,141,121.0,3.662890,3.787648,3.044034,21,-0.466867,20.924999,3.662890,10.833333,0,0,0,True,1


# <span style="color:blue; font-weight:bold;">2. traitements des valeurs manquantes</span>

Les **données réelles** contiennent souvent des lignes incomplètes (ex : retard non enregistré, aéroport manquant, etc.). Si elles ne sont pas gérées, ces lacunes peuvent introduire des **biais** dans les modèles de régression.

### Notre Démarche

Nous devons donc procéder méthodiquement :

1.  **Identifier** : Repérer les colonnes contenant des valeurs manquantes.
2.  **Filtrer (Élagage)** : **Supprimer** les lignes incomplètes (ou "observations partielles") pour éviter les biais dans l'apprentissage.
3.  **Garantie de Qualité** : Conserver **uniquement les observations complètes**.

Cette étape fondamentale garantit que nos modèles reçoivent un ensemble de données **fiable et complet** pour des prédictions précises.

In [5]:
print(df.head())
print(df.describe())

   depdelay  arrdelay    origin      dest  uniquecarrier  scheduledhour  \
0        -2     -11.0  5.875096  5.678700       5.262673             17   
1       -10      -1.0  7.817186  2.527276       8.747081              8   
2        23      13.0  5.994635  3.972538       8.646316              8   
3        -4     -14.0 -0.443869  3.599142       6.162031              7   
4       141     121.0  3.662890  3.787648       3.044034             21   

     tailnum  numflights  origincityname  windspeed  windgustdummy  raindummy  \
0   4.984410   15.471000        5.875096   9.133333              1          0   
1   9.403636   20.132999        5.577255  13.000000              0          0   
2  10.882033   17.000999        5.994635   7.166667              1          0   
3   6.645973   17.858999       -0.443869  21.000000              0          0   
4  -0.466867   20.924999        3.662890  10.833333              0          0   

   snowdummy  is_holiday_season  est_retarde  
0          0   

In [7]:
df = df.dropna()
df.drop('est_retarde', axis=1, inplace=True)
print(df.shape)

(781081, 14)


# <span style="color:blue; font-weight:bold;">3. Séparation entre variables explicatives et variable cible</span>

La variable que nous cherchons à prédire est :
*  **`arrdelay`** (Retard à l’arrivée, en minutes).

Les variables utilisées pour entraîner le modèle et expliquer le retard à l'arrivée incluent :

* `depdelay` (Retard au départ)
* `year` (Année du vol)
* `dayofweek` (Jour de la semaine)
* `scheduledhour` (Heure prévue du vol)
* `origairportid` (Identifiant de l'aéroport d'origine)
* `destairportid` (Identifiant de l'aéroport de destination)
* ... et **toutes les autres colonnes** pertinentes du jeu de données.

### Objectif de l'Étape

L'objectif principal est de **préparer ces données (nettoyage, encodage, normalisation)** afin d'obtenir des jeux d'entraînement et de test optimaux pour l'**entraînement des modèles de régression**.

In [9]:
y = df["arrdelay"]
X = df.drop(["arrdelay"], axis=1)

<h3> Division des données en ensemble d’entraînement et de test

In [11]:
from sklearn.model_selection import train_test_split

X_train , X_test , y_train , y_test = train_test_split(X,y, test_size=0.2 , random_state=42)


 # <span style="color:blue; font-weight:bold;">4.Application des algorithmes de régression</span>

## <font color="red">Régression Linéaire</font>

La **Régression Linéaire** est un algorithme de base utilisé pour prédire une **valeur numérique**.

Dans notre contexte, cet algorithme permet de prédire le **retard à l'arrivée** (`arrdelay`) d'un vol en utilisant des variables comme le retard au départ, le jour de la semaine, l'heure prévue, l'aéroport d'origine/destination, etc.

L'objectif est de trouver la **relation linéaire** optimale entre ces variables et la variable cible, selon l'équation suivante :

$$\text{arrdelay} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots + \beta_n x_n$$

Où les **coefficients $\beta_i$** indiquent l'influence de chaque variable ($x_i$) sur le retard.


In [13]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("RMSE :", rmse)
print("R² :", r2)

MAE : 9.715026290783813
RMSE : 14.433932801408385
R² : 0.8919324467879927


<h4> exemple de test 

In [29]:
test_example = {
    "depdelay": [20],
    "origin": ['CAS'],
    "dest": ['CMN'],
    "uniquecarrier": ['AT'],
    "scheduledhour": [16],
    "tailnum": ['T001'],
    "numflights": [2],
    "origincityname": ['Casablanca'],
    "windspeed": [10],      
    "windgustdummy": [0],
    "raindummy": [0],
    "snowdummy": [0],
    "is_holiday_season": [0]
}

df_test = pd.DataFrame(test_example)


df_test = pd.get_dummies(df_test, columns=['origin', 'dest', 'uniquecarrier', 'tailnum', 'origincityname'], drop_first=True)


for col in X.columns:
    if col not in df_test.columns:
        df_test[col] = 0
df_test = df_test[X.columns] 

# Prédiction
predicted_arrival_delay = model.predict(df_test)
print("Retard d'arrivée prédit :", predicted_arrival_delay[0], "minutes")

Retard d'arrivée prédit : 2.6213735049460567 minutes


##### Évaluation du Modèle de Régression Linéaire

| Métrique | Valeur Obtenue | Interprétation |
| :--- | :--- | :--- |
| **MAE** (Mean Absolute Error) | **8.79 minutes** | En moyenne, l'erreur absolue entre la prédiction et la valeur réelle est de **$\pm$ 9 minutes**. |
| **RMSE** (Root Mean Squared Error) | **12.67 minutes** | Cette valeur pénalise davantage les erreurs importantes. Les erreurs quadratiques peuvent atteindre environ **13 minutes**. |
| **R²** (Coefficient de Détermination) | **0.887** | Le modèle explique environ **89%** de la variance totale des retards d'arrivée. |

---

### Conclusion

Le coefficient **R² élevé (0.887)** est un indicateur de **bonnes performances** du modèle. Il prédit correctement les retards d'arrivée 

## <font color="red">Reseaux bayessiens</font>

In [32]:
from sklearn.linear_model import BayesianRidge
model = BayesianRidge()
model.fit(X_train, y_train)

# Prédiction
y_pred = model.predict(X_test)

# Calcul des métriques
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Prédictions :", y_pred)
print("MSE :", mse)
print("RMSE :", rmse)
print("MAE :", mae)
print("R² :", r2)

Prédictions : [ 26.67989308 -11.08856723 133.88612124 ...  -3.33852688  98.69942829
  -4.72155047]
MSE : 208.33793888132143
RMSE : 14.433916269721168
MAE : 9.715012115774144
R² : 0.8919326943349349


## <font color="red">K-Nearest Neighbors (KNN)</font>
L’algorithme K-Nearest Neighbors (KNN) est un algorithme non paramétrique.

Pour prédire un retard de vol, KNN regarde les $K$ vols les plus similaires (les plus proches dans l’espace des features), puis calcule la moyenne des retards de ces $K$ voisins.

<h2> normalisation

In [34]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(X_train)
X_train_norm = scaler.transform(X_train)
X_test_norm = scaler.transform(X_test)

<h2> application de l'algorithme 

In [ ]:

from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import time

print("🔄 Début du processus KNN...")
time.sleep(1)

# -------------------------------
# 1. Paramètres
# -------------------------------
print("📌 Chargement des hyperparamètres...")
param_dist = {
    'n_neighbors': [3, 5, 7],
    'weights': ['uniform', 'distance'],
    'p': [1, 2]
}
time.sleep(1)

# -------------------------------
# 2. Préparation RandomizedSearch
# -------------------------------
print("⚙️ Préparation du RandomizedSearchCV (cela peut prendre du temps)...")
rand_search = RandomizedSearchCV(
    KNeighborsRegressor(),
    param_distributions=param_dist,
    n_iter=3,
    cv=2,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
)

# -------------------------------
# 3. Fit
# -------------------------------
print("🚀 Début de l'entraînement GridSearch sur KNN...")
start_time = time.time()

rand_search.fit(X_train_norm, y_train)

elapsed = time.time() - start_time
print(f"✅ RandomizedSearch terminé en {elapsed:.2f} secondes")

print("🏆 Meilleurs paramètres trouvés :", rand_search.best_params_)

# -------------------------------
# 4. Prédiction
# -------------------------------
print("🔮 Prédiction en cours sur X_test...")
y_pred = rand_search.best_estimator_.predict(X_test_norm)

print("📊 Calcul des métriques...")
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("\n📈 **Résultats finaux KNN**")
print("MSE  :", mse)
print("RMSE :", rmse)
print("R²   :", r2)

print("\n✨ Fin du programme.")


🔄 Début du processus KNN...
📌 Chargement des hyperparamètres...
⚙️ Préparation du RandomizedSearchCV (cela peut prendre du temps)...
🚀 Début de l'entraînement GridSearch sur KNN...


In [36]:
import numpy as np

# On prend un échantillon de 30000 lignes
sample_size = 30000
idx = np.random.choice(len(X_train_norm), size=sample_size, replace=False)

X_sample = X_train_norm[idx]
y_sample = y_train.iloc[idx] if hasattr(y_train, "iloc") else y_train[idx]

print("Sample shape :", X_sample.shape)


from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_neighbors': [3, 5, 7],
    'weights': ['uniform', 'distance'],
    'p': [1, 2]
}

rand_search = RandomizedSearchCV(
    KNeighborsRegressor(),
    param_distributions=param_dist,
    n_iter=3,
    cv=2,
    n_jobs=-1,
    scoring='neg_mean_squared_error'
)


rand_search.fit(X_sample, y_sample)

print("BEST PARAMS :", rand_search.best_params_)



best_model = rand_search.best_estimator_
best_model.fit(X_train_norm, y_train)



y_pred = best_model.predict(X_test_norm)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MSE :", mse)
print("RMSE :", rmse)
print("R² :", r2)


Sample shape : (30000, 13)
BEST PARAMS : {'weights': 'uniform', 'p': 2, 'n_neighbors': 5}
MSE : 285.7228987882241
RMSE : 16.903339870813227
R² : 0.8517922179481454


## Modèle K-Nearest Neighbors (KNN) : Résumé de l'Analyse

### Performance
* **R² = 0.85 :** Le modèle est **correct** (explique 80% de la variance des retards).
* **RMSE = 16.85 minutes :** L'erreur de prédiction moyenne de **17 minutes** est **trop élevée** pour une application pratique de prédiction de retard.
* **Meilleurs Paramètres :** Le modèle utilise les **5 voisins les plus proches** ($k=3$) et accorde plus de poids à la **distance euclidienne** pour la prédiction.

### Conclusion sur l'Adaptabilité
Le modèle KNN est **inadapté** a cette  *dataset*  car il est tres lent .
.

## <font color="red"> Random Forest (Forêt Aléatoire)</font>

Le **Random Forest** est un algorithme d'apprentissage automatique de type **ensemble** (*méthode d'agrégation*). Il combine les résultats de plusieurs **arbres de décision** indépendants pour obtenir une prédiction plus **précise** et **robuste**.

Pour prédire le retard d'un nouveau vol :

1.  Chaque arbre de décision dans la forêt donne une estimation du retard.
2.  L'algorithme calcule ensuite la **moyenne** de toutes les prédictions individuelles des arbres pour fournir la prédiction finale.


In [62]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score

print("Définition du modèle Random Forest et des hyperparamètres pour RandomizedSearchCV...")
rf = RandomForestRegressor(random_state=42, n_jobs=-1)

param_dist = {
    'n_estimators': [10, 50 , 100],
    'max_depth': [5, 15,None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
}

random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=5,
    cv=2,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

print("Début de l'entraînement avec RandomizedSearchCV...")
random_search.fit(X_train, y_train)
print("RandomizedSearchCV terminé !")
print("Meilleurs paramètres :", random_search.best_params_)

# -----------------------------
# Prédictions
# -----------------------------
print("Début des prédictions sur le set de test...")
best_model = random_search.best_estimator_
y_pred = best_model.predict(X_test)  

# -----------------------------
# Évaluation
# -----------------------------
print("Évaluation du modèle...")
mse = mean_squared_error(y_test, y_pred)
rmse = mse**0.5
r2 = r2_score(y_test, y_pred)

print("MSE :", mse)
print("RMSE :", rmse)
print("R² :", r2)


Définition du modèle Random Forest et des hyperparamètres pour RandomizedSearchCV...
Début de l'entraînement avec RandomizedSearchCV...
Fitting 2 folds for each of 5 candidates, totalling 10 fits
RandomizedSearchCV terminé !
Meilleurs paramètres : {'n_estimators': 50, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': 15}
Début des prédictions sur le set de test...
Évaluation du modèle...
MSE : 206.89357465819865
RMSE : 14.383795558134112
R² : 0.8926819028124211


## Interprétation des Résultats du Random Forest

**Meilleurs paramètres :** `{'n_estimators': 40, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_depth': 15}`
*MSE : 205.7473* | *RMSE : 14.3439* | *$R^2$ : 0.8933*



Le modèle **Random Forest** démontre une ** bonne performance ** pour la prédiction des retards.

* Le **Coefficient de Détermination ($R^2$) de 0.8933** indique que le modèle explique près de **90 %** de la variance observée dans les retards de vol.
* L'erreur de prédiction moyenne (**RMSE**) est de **14.34 minutes**.

**En résumé :** Le modèle est **robuste et très précis**, fournissant des estimations de retard fiables avec une marge d'erreur moyenne acceptable.

## <font color="red"> Multilayer Perceptron (MLP)</font>

Le Multilayer Perceptron (MLP) est un réseau de neurones artificiels fondamental du Deep Learning.

Structure : Il est organisé en trois types de couches : entrée, cachées et sortie.

Apprentissage : Il utilise les couches cachées pour découvrir et modéliser des relations complexes et non linéaires dans les données.

Régression : Pour prédire le retard de vol, le MLP ajuste les poids de ses connexions pour minimiser l'erreur.

In [60]:
from sklearn.neural_network import MLPRegressor
print("Définition du modèle MLP et des hyperparamètres pour RandomizedSearchCV...")
mlp = MLPRegressor(random_state=42, max_iter=200)

param_dist = {
    'hidden_layer_sizes': [(50,), (100,), (50,50)],   
    'activation': ['relu', 'tanh'],
    'solver': ['adam'],                               
    'learning_rate_init': [0.001, 0.01],
    'batch_size': [256, 512, 1024],                  
}

random_search = RandomizedSearchCV(
    estimator=mlp,
    param_distributions=param_dist,
    n_iter=10,
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

# Entraînement

print("Début de l'entraînement avec RandomizedSearchCV...")
random_search.fit(X_train_norm, y_train)
print("RandomizedSearchCV terminé !")
print("Meilleurs paramètres :", random_search.best_params_)


# Prédictions

print("Début des prédictions sur le set de test...")
best_model = random_search.best_estimator_
y_pred = best_model.predict(X_test_norm)
print("Prédictions terminées !")


#  Évaluation

print("Évaluation du modèle...")
mse = mean_squared_error(y_test, y_pred)
rmse = mse**0.5
r2 = r2_score(y_test, y_pred)

print("MSE :", mse)
print("RMSE :", rmse)
print("R² :", r2)



Définition du modèle MLP et des hyperparamètres pour RandomizedSearchCV...
Début de l'entraînement avec RandomizedSearchCV...
Fitting 3 folds for each of 10 candidates, totalling 30 fits
RandomizedSearchCV terminé !
Meilleurs paramètres : {'solver': 'adam', 'learning_rate_init': 0.001, 'hidden_layer_sizes': (50, 50), 'batch_size': 1024, 'activation': 'relu'}
Début des prédictions sur le set de test...
Prédictions terminées !
Évaluation du modèle...
MSE : 204.21031375978282
RMSE : 14.290217414713565
R² : 0.8940737413668641


C:\Users\hp\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:788: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")
